# 9.1 Regular Expression

**Prerequisites:** 2.1 Strings, 08 File Handling  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- What a regex is, and the history behind the syntax
- `compile`, and the `Pattern` object
- `match`, `search`, `finditer`, `findall`, **`fullmatch`**, `split`, `sub`
- **The `Match` object** — `groups()`, `groupdict()`, `span()`, and the walrus idiom
- The backslash plague, and why patterns are **always** raw strings
- Character classes, quantifiers, greediness, boundaries
- All the compilation flags, including `re.ASCII` and the bytes-only `re.LOCALE`
- Grouping, backreferences, named and non-capturing groups
- Lookahead and lookbehind
- 🔴 **Catastrophic backtracking (ReDoS)** and the three fixes
- Performance, and when a `str` method is the better answer

---

## Regular Expression(RegEx)
- A RegEx is a sequence of characters that defines a search pattern for searching something in a given text.
- A regular expression often called pattern is an expression used to specify a set of characters required for a particuar purpose. 
- For ex. Matching set of strings like {file, file1, file2}, RegEx can be written as file(1|2)? or file/d?
    - We say that this pattern matches each of the three strings. [check?](https://regexr.com/48om5)


## Maths of Regular Expressions
- The concept of **Regular Expressions** originated from **[Regular Languages](https://en.wikipedia.org/wiki/Regular_language)**. 
- **Regular Expressions** describe **Regular Languages** in **[Formal Language Theory](https://en.wikipedia.org/wiki/Formal_language)**.
<br/><br/>
- ***Formal Language Theory:***
    - In mathematics, computer science, and linguistics, a **formal language** consists of words whose letters are taken from an alphabet and are **well-formed according to a specific set of rules**. 
    - The field of formal language theory studies primarily the purely syntactical aspects of such languages—that is, their internal structural patterns.
- ***Regular Languages:*** 
    - A regular language is a category of **formal languages** which can be expressed using a regular expression.!
    - <img src='./Image/9.1 Image a.png'>

- **NOTE:** Today, many regular expressions engines provided by modern programming languages are augmented with features that allow recognition of languages that <span style="color:red;">**cannot**</span> be expressed by a classic regular expression!


## Need for RegEx
- Some important usages of RegEx are:
    - 1) Look for a pattern appearance in a piece of text. for example, 
        - Check if either the word "color" or the word "colour" appears in a document with just **one scan**
        - Search in Log data for specific date and time:
            - <img src='./Image/9.1 Image b.png' width=65% height=40%/>
    <br/><br/>
    - 2) Check if an input is in accordance with a given pattern. For ex.
        - We can check whether a value entered in a HTML formulary is a valid e-mail address. 
            - Ex. schx#mail.com, schoo@gmail.com, scx@fx
    <br/><br/>
    - 3) Extract specific portions of a text. For ex. 
        - Extract the postal code of an address for a file.
        - Collecting information of persons from a particular area based on adddress pincode.
        - Extracting specific content from a website in Web Scrapping.
    <br/><br/>
    - 4) Replace portions of text. For ex. 
        - Change any appearance of "color" or "colour" with "red".
        - Want to update a particular detail from student database for all students. 
    <br/><br/>
    - 5) Split a larger text into smaller pieces. For ex.  
        - Splitting a text by any appearance of the dot, comma, or newline characters.

## 4. A brief history of Regular Expressions

> *The story begins with a neuroscientist and a logician who together tried to understand how the human brain could produce complex patterns using simple cells that are bound together.*

- In 1943, neurophysiologists ***Warren McCulloch*** and ***Walter Pitts*** published ***"A logical calculus of the ideas immanent in nervous activity"***. This paper not only represented the beginning of the regular expressions, but also proposed the first mathematical model of a neural network.


- In 1956, ***Stephen Kleene*** wrote the paper ***"Representation of events in nerve nets and finite automata"***, where he coined the terms **regular sets** and **regular expressions** and presented a simple algebra.


- In 1968, the Unix pioneer ***Ken Thompson***  took Kleene's work and extended it, publishing his studies in the paper ***"Regular Expression Search Algorithm"***. Ken Thompson's work didn't end in just writing a paper. He also implemented Kleene’s notation in the editor ***QED***. The aim was that the user could do advanced pattern matching in text files. The same feature appeared later on in the editor ***ed***.

> To search for a Regular Expression in ed you wrote `g/<regular expression>/p` The letter g meant global search and p meant print the result. The command — `g/re/p` — resulted in the standalone program grep, released in the fourth edition of Unix 1973.<br><span style="color:red;">However, **grep** didn’t have a complete implementation of regular expressions.</span>

- In 1979, ***Alfred Aho*** developed ***egrep (extended grep)*** in the seventh edition of Unix. The program egrep translated any regular expressions to a corresponding [DFA](https://en.wikipedia.org/wiki/Deterministic_finite_automaton).


- In 1987, ***Larry Wall*** created the scripting language ***Perl***. Regular Expressions are seamlessly integrated in Perl, even with its own literals. Hence, Perl pushed the regular expressions to the mainstream. The implementation in Perl went forward and added many modifications to the original regular expression syntax, creating the so-called ***Perl flavor***.

### Some other worth mentioning milestones

- The IEEE thought their POSIX standard has tried to standardize and give better Unicode support to the regular expression syntax and behaviors. This is called the ***POSIX flavor*** of the regular expressions.


- In late 1980s, ***Henry Spencer*** wrote ***"regex"***, a widely used software library for regular expressions in C programming langauge.


### Here is a brief timeline to summarize...

<img src='./Image/9.1 Image c.png'>

### Regex today

- It was the rise of the web that gave a big boost to the Perl implementation of regex, and that's where we get the modern syntax of regular expressions today; it really comes from Perl. `Apache`, `C`, `C++`, `the .NET languages`, `Java`, `JavaScript`, `MySQL`, `PHP`, `Python`, `Ruby` all of these are endeavoring to be Perl-compatible languages and programs. There's also a library called the `PCRE` library that stands for Perl-Compatible Regular Expression library.


- Today, the standard Python module for regular expressions—`re`—supports only Perl-style regular expressions. There is an [effort](https://pypi.python.org/pypi/regex) to write a new regex module with better POSIX style support. This new module is intended to replace Python's `re` module implementation eventually. 

## Understanding the RegEx Syntax:
- A regex is a simple sequence of characters. The components of a regex pattern are:
    - <img src='./Image/9.1 Image d.png'>
    - **literals (ordinary characters)**: These characters carry no special meaning and are processed as it is.
    - **metacharacters (special characters)**: These characters carry a special meaning and processed in some special way.
- Hence, a regular expression can be formed by using the mix of literals, meta-characters, special sequences, and sets.    

- Let's start with a simple example.
    - Consider that we have got the list of several filenames in a folder.
    ```
    file1.xml
    file1.txt
    file2.txt
    file15.xml
    file5.docx
    file60.txt
    file5.txt
    ```
    - And we want to filter out only those filenames which follow a specific pattern, i.e.  `file<one or more digits>.txt`.
    - Let's try to do this on an online tool to learn, build, & test Regular Expressions (RegEx / RegExp), 
        - [RegExr](https://regexr.com).
        - So, the regular expression we need here is: `file\d+\.txt`

- This expression can be understood as follows:
    - <img src='./Image/9.1 Image e.png'>
    - `file` is a substring of literals which are matched with the input as it is.
    - `\d` is a metacharacter which instructs the software to match this position with a digit (0-9).
    - `+` is also a metacharacter which instructs the software to match one or more iterations of the preceeding character (`\d` in this case)
    - `\.` is a literal. `.` is a metacharacter but we want to use it as a literal in this case. Hence, we escape it using `\` character.
    - `txt` is a substring of literals which are matched with the input as it is.



## RegEx in Python
- The **[re](https://docs.python.org/3/howto/regex.html)** module provides an interface to the regular expression engine.
    - It allows us to **compile regular expressions into objects and then perform matches with them**.

In [ ]:
import re

In [ ]:
from utils import highlight_regex_matches as highlight

In [ ]:
# !pip install colorama

### 1. Compiling Regular Expressions
- Regular expressions are **compiled** into `Pattern` objects, which have methods for various operations such as searching for pattern matches or performing string substitutions.
- **Syntax:** `re.compile(pattern, flags=0)`
    - Compile a regular expression pattern, returning a pattern object.
    - The regular expression is passed to `re.compile()` as a **string**. 

> Regular expressions are handled as strings because regular expressions aren't part of the core Python language, and no special syntax was created for expressing them. 

> Regular expression patterns are compiled into a series of bytecodes which are then executed by a matching engine written in C.

In [ ]:
pattern = re.compile("hello")
print(pattern)

- `re.compile()` also accepts an optional `flags` argument, used to enable various special features and syntax variations. [More about flags](http://xahlee.info/python/python_regex_flags.html)
<br/><br/>
- In the example below, we use the flag `re.I` (short for `re.IGNORECASE`) to ignore letter case in the regex pattern.

In [ ]:
pattern = re.compile("hello", flags=re.I)
print(pattern)

### 2. Performing Matches
- We created a `Pattern` object representing a compiled regular expression using `re.compile()` method.
- Pattern objects have several methods and attributes. Here is the list of different methods used for performing matches:

<table style="border: 1px solid black; font-size:15px;">
<thead>
    <th>Method/Attribute</th>
    <th>Purpose</th>
</thead>
    
<tbody>
<tr>
    <td>match()</td>
    <td>Determine if the RE matches at the beginning of the string.</td>
</tr>
    
<tr>
    <td>search()</td>
    <td>Scan through a string, looking for any location where this RE matches.</td>
</tr>

<tr>
    <td>finditer()</td>
    <td>Find all substrings where the RE matches, and returns them as an iterator.</td>
</tr>

<tr>
    <td>findall()</td>
    <td>Find all substrings where the RE matches, and returns them as a list.</td>
</tr>
</tbody>
</table>

### a. `match(string[, pos[, endpos]])`
- A match is checked only at the beginning (by default).
- Checking starts from `pos` index of the string. (default is 0)
- Checking is done until `endpos` index of string. `endpos` is set as a very large integer (by default).
- Returns `None` if no match found.
- If a match is found, a `Match` object is returned, containing information about the match: where it starts and ends, the substring it matched, and more.

In [ ]:
txt= 'Good afternoon all, good to see you here.'
pattern = re.compile("good", flags=re.I)
print(pattern)

In [ ]:
match = pattern.match(txt)
print(match) #return match object
print(match.span())
print(match.start())
print(match.end())

In [ ]:
match = pattern.match(txt, pos=5)
print(match) # if no match found

In [ ]:
match = pattern.match(txt, pos=20)
print(match)

### b. `search(string[, pos[, endpos]])`
- A match is checked throughtout the string.
- Same behaviour of `pos` and `endpos` as the `match()` function.
- Returns `None` if no match found.
- If a match is found, a `Match` object is returned.

In [ ]:
txt= 'Good afternoon all, good to see you here.'
pattern = re.compile("good", flags=re.I)
match = pattern.search(txt)
print(match)

In [ ]:
match = pattern.search(txt, pos=5)
print(match)

### c. `finditer(string[, pos[, endpos]])`
- Finds **all non-overlapping substrings** where the match is found, and returns them as an iterator of the `Match` objects.
- Same behaviour of `pos` and `endpos` as the `match()`, `search()` and `findall()` function.

In [ ]:
txt= 'Good afternoon all, good to see you here.'
pattern = re.compile("good", flags=re.I)
matches = pattern.finditer(txt)
for match in matches:
    print(match)
    print(match.span())

### d. `findall(string[, pos[, endpos]])`
- Finds **all non-overlapping substrings** where the match is found, and returns them as a list.
- Same behaviour of `pos` and `endpos` as the `match()` and `search()` function.

In [ ]:
txt= 'Good afternoon all, good to see you here.'
pattern = re.compile("good", flags=re.I)
matches = pattern.findall(txt)
print(matches)

> By now, you must have noticed that `match()`, `search()` and `finditer()` return `Match` object(s) where as `findall()` returns a list of strings.

In [ ]:
highlight(pattern, txt)

### f. `fullmatch(string[, pos[, endpos]])`

The third matching function, and the one most people should be using for **validation**.

| Function | Anchors | `r"\d+"` against `"123abc"` |
|---|---|---|
| `match()` | Start only | ✅ matches `"123"` |
| `search()` | Nowhere — scans | ✅ matches `"123"` |
| **`fullmatch()`** | **Both ends** | ❌ no match |

> **Version note:** `fullmatch` was added in **Python 3.4**.

The classic bug it prevents: using `match()` to validate, and accepting
`"123abc"` as a valid number because `match` only anchored the *start*. Writing
`re.match(r"\d+$", s)` works too, but `fullmatch` says what you mean.

### The `Match` object

Every successful match returns a `Match`, and it holds more than the matched text.

In [ ]:
import re

candidates = ["123", "123abc", "abc123", ""]
pattern = r"\d+"

print(f"pattern {pattern!r}\n")
print(f"  {'input':<10} {'match':<12} {'search':<12} {'fullmatch'}")
for s in candidates:
    m = "yes " + repr(re.match(pattern, s).group()) if re.match(pattern, s) else "no"
    se = "yes " + repr(re.search(pattern, s).group()) if re.search(pattern, s) else "no"
    f = "yes" if re.fullmatch(pattern, s) else "no"
    print(f"  {s!r:<10} {m:<12} {se:<12} {f}")

print("""
  ^ '123abc' is the important row. match() and search() both succeed,
    because neither anchors the END. Only fullmatch() rejects it.
""")

# ---- Validation done properly ----
def is_valid_pin(value: str) -> bool:
    return re.fullmatch(r"\d{4}", value) is not None

for pin in ["1234", "12345", "12a4", "1234 ", ""]:
    print(f"  {pin!r:<8} -> {is_valid_pin(pin)}")

In [ ]:
import re

# ---- A Match object carries far more than the matched text ----
pattern = re.compile(r"(?P<user>\w+)@(?P<domain>[\w.]+)")
m = pattern.search("contact aditya@example.com for details")

print("group()      :", m.group())         # the whole match
print("group(1)     :", m.group(1))        # first group, by number
print("group('user'):", m.group("user"))   # by name
print("groups()     :", m.groups())        # all groups as a tuple
print("groupdict()  :", m.groupdict())     # named groups as a dict
print("span()       :", m.span())          # (start, end)
print("start(), end():", m.start(), m.end())
print("string       :", m.string[m.start():m.end()])
print("re           :", m.re.pattern)

# ---- groupdict() is why named groups are worth the extra characters ----
LOG = re.compile(
    r"(?P<date>\d{4}-\d{2}-\d{2})\s+"
    r"(?P<level>[A-Z]+)\s+"
    r"(?P<message>.+)"
)

for line in ["2024-03-15 ERROR db connection refused",
             "2024-03-16 INFO  service started"]:
    record = LOG.match(line).groupdict()
    print(f"\n  {record}")
    print(f"  level={record['level']:<6} on {record['date']}")

# ---- The walrus operator makes the common shape read cleanly ----
lines = ["2024-03-15 ERROR db down", "not a log line", "2024-03-16 INFO ok"]

print()
for line in lines:
    if match := LOG.match(line):                    # 3.8+ (see 1.4)
        print(f"  parsed : {match['level']}")       # m[...] is shorthand for m.group(...)
    else:
        print(f"  skipped: {line!r}")

### e. Split using RegEx
- In almost every language, you can find the split operation in strings. 
- The big difference is that the split in the `re` module is more powerful due to which you can use a regex. 
- So, in this case, the string is split based on the matches of the pattern.
- **Syntax:** `split(string[, maxsplit])`
    - Every pattern object has a `split()` method which splits the input string at all positions where a match is found.
    - `maxsplit` is an optional argument (default value 0) which specifies the max no. of splits that can take place. `0` value means there is no limit on the no. of splits.
    - Pattern match is not included in any of the substrings obtained after splitting.
    

### Example 1
- Let us try to split a string to get individual lines in it.

In [ ]:
txt = """Beautiful is better than ugly.
Explicit is better than implicit.
Simple is better than complex.
Complex is better than complicated."""
pattern = re.compile("\n")
list_= pattern.split(txt)
print(list_)

### Example 2
- Let us try one more example in which we want to get all the words in the given text.

In [ ]:
pattern = re.compile(r"\W")
after= pattern.split(txt)
print(after)

### Example 3
- What is we want only first 3 words? We need to split only 3 times in this case, which can be done by setting the value of `maxsplit` as 3.

In [ ]:
pattern = re.compile(r"\W")
after= pattern.split(txt,  maxsplit=3)
print(after)

### NOTE:
- It is not mandatory to create a `Pattern` object explicitly using `re.compile()` method in order to perform a regex operation.
- We can direclty use the module level functions such as:
    - `re.match(pattern, string, flags=0)`
    - `re.search(pattern, string, flags=0)`
    - `re.findall(pattern, string, flags=0)`
    - `re.finditer(pattern, string, flags=0)`
    - `re.split(pattern, string, maxsplit=0)`
- In a module level function, we can simply pass a **string** as our **regex pattern**.
- Look at the examples shown below:

In [ ]:
txt= 'Good afternoon all, good to see you here.'

In [ ]:
re.match("good", txt, flags=re.I)

In [ ]:
re.search("good", txt, flags=re.I)

In [ ]:
for match in re.finditer("good", txt, flags=re.I):
    print(match)

In [ ]:
re.findall("good", txt, flags=re.I)

In [ ]:
# 🔴 The original was:  re.split(' ', txt, 2)
#
# Version note: passing `maxsplit` (and `count` for re.sub) POSITIONALLY is
# deprecated as of Python 3.13 and will become an error. Pass it by keyword.

print("keyword form :", re.split(r" ", txt, maxsplit=2))

import warnings
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    re.split(r" ", txt, 2)
    if caught:
        print("positional   :", caught[0].category.__name__, "-", caught[0].message)
    else:
        print("positional   : no warning on this version")

# The same applies to re.sub's `count`
print("\nre.sub keyword:", re.sub(r"o", "0", "foo boo", count=1))

### Metacharacters:
- In RegEx, there are 12 metacharacters:
    - Backslash `\`
    - Caret `^`
    - Dollar sign `$`
    - Dot `.`
    - Pipe symbol `|`
    - Question mark `?`
    - Asterisk `*`
    - Plus sign `+`
    - Opening parenthesis `(`
    - Closing parenthesis `)`
    - Opening square bracket `[`
    - The opening curly brace `{`


- **NOTE:** In order to treat a metacharacter like a literal, we need to **escape** it using `\` character.

In [ ]:
txt = "This book costs $15."
# match $15
pattern = re.compile("$15") #no matches found
match= pattern.search(txt)
print(match)

In [ ]:
# We need to escape '$' metacharacter
pattern = re.compile(r"\$15")
match= pattern.search(txt)
print(match)

## Backslash Plague

- Consider a file of containing some Windows style directory addresses in which we have to find `C:\Windows\System32` substring.

In [ ]:
txt = r"""
C:\Windows
C:\Python
C:\Windows\System32
"""

In [ ]:
pattern = re.compile(r"C:\Windows\System32") 
match= pattern.search(txt)
print(match)

#### Why are no matches found for above pattern?
- Regex Engine is treating `\` as metacharacters, whereas we intend to treat it like a literal.

#### Solution???
- We need to escape the metacharacters. 
    - A metacharacter can be escaped by putting a `\` before it.

In [ ]:
pattern = re.compile("C:\\Windows\\System32")
match= pattern.search(txt)
print(match)

In [ ]:
print("C:\\Windows\\System32") # Because python interprets uses '\\' escape sequence as '\'

#### Still no match found. Why???
- Because `\` is used as an escape at two different levels. 
    - First, the Python interpreter itself performs substitutions for `\` before the `re` module ever sees the pattern string. 
        - For instance, `\n` is converted to a newline character, `\t` is converted to a tab character, etc. 
    - Finally, `re` reads the substituted pattern string and will apply its own substitutions for `\` character. 
- Hence, to use `\` as a **literal**, we first escape `\` with `\\` for python interpreter and then escape `\\` as `\\\\` for regex engine.

In [ ]:
pattern = re.compile("C:\\\\Windows\\\\System32")
match= pattern.search(txt)
print(match)

#### Can we use 2 backslashes instead of 4 here?
- Yes. By using **raw-strings**, we do not need to put escapes at first level. 

> Python raw strings are represented as ***r"your string"***. 
      In raw strings, no escaping is required as escape sequences like `\n`, `\t`, etc are not processed.

In [ ]:
pattern = re.compile(r"C:\\Windows\\System32")
match= pattern.search(txt)
print(match)

#### Do we really need to use 2 backslashes?
- If you are **not using any metacharacters** in your regex pattern, you can use `re.escape()` method to escape all the characters in pattern except ASCII letters, numbers and '_'.

In [ ]:
pattern = re.compile(re.escape(r"C:\Windows\System32"))
match= pattern.search(txt)
print(match)

## Character Classes or Character Sets
- The **character sets** allow us to define a character sequence that will match if any of the defined characters on the set is present.
- To define a character sets, we should use the opening square bracket metacharacter `[`, then any accepted characters, and finally close with a closing square bracket `]`.


### Example 1:
- Consider an example below where we have messed up between `license` and `licence` spellings and want to find all occurances of `license`/`licence` in the text.

In [ ]:
txt = """
Yesterday, I was driving my car without a driving licence. 
The traffic police stopped me and asked me for my license. 
I told them that I forgot my licence at home. 
"""
pattern = re.compile("licen[cs]e")
match = pattern.findall(txt)
print(match)

<img src='./Image/9.1 Image f.png' width=40% height=50%/>

In [ ]:
highlight(pattern, txt)

## Character Sets Range:
- It is also possible to use the range of a character. 
- This is done by leveraging the hyphen symbol (-) between two related characters. For ex.
    - To match any lowercase letter we can use `[a-z]`.
    - To match any single digit we can define the character set `[0-9]`.
    - To match any lowercase or uppercase letter we can use `[a-zA-Z]`.
    - Character Sets Range `[0-385]` means `[012385]`


### Example 1:

In [ ]:
txt = """
The first season of Indian Premiere League (IPL) was played in 2008. 
The second season was played in 2009 in South Africa. 
Last season was played in 2018 and won by Chennai Super Kings (CSK).
CSK won the title in 2010 and 2011 as well.
Mumbai Indians (MI) has also won the title 3 times in 2013, 2015 and 2017.
"""
#retrieve all the years from the given text
pattern = re.compile("[1-9][0-9][0-9][0-9]")
matches= pattern.findall(txt)
print(matches)

In [ ]:
highlight(pattern, txt)

- There is another possibility — the **negation of ranges**. 
- We can invert the meaning of a character set by placing a caret (`^`) symbol right after the opening square bracket metacharacter (`[`). For ex.
    - To match all characters other than lowercase letters, we can use `[^a-z]`

In [ ]:
txt = """
The first season of Indian Premiere League (IPL) was played in 2008. 
The second season was played in 2009 in South Africa. 
Last season was played in 2018 and won by Chennai Super Kings (CSK).
CSK won the title in 2010 and 2011 as well.
Mumbai Indians (MI) has also won the title 3 times in 2013, 2015 and 2017.
"""
# filter all vowel charcaters:
pattern = re.compile("[^aeiou]")
matches= ''.join(pattern.findall(txt))
print(matches)

In [ ]:
highlight(pattern, txt)

## Predefined Character Classes
- There exist some predefined character classes which can be used as a shortcut for some frequently used classes.

<table style="border: 1px solid black; font-size:15px;">
<thead>
    <th>Element</th>
    <th>Description</th>
</thead>
    
<tbody>
<tr>
    <td>.</td>
    <td>This element matches any character except newline</td>
</tr>

<tr>
    <td>\d</td>
    <td>This matches any decimal digit; this is equivalent to the class [0-9]</td>
</tr>

<tr>
    <td>\D</td>
    <td>This matches any non-digit character; this is equivalent to the class [^0-9]</td>
</tr>

<tr>
    <td>\s</td>
    <td>This matches any whitespace character; this is equivalent to the class
[ \t\n\r\f\v]</td>
</tr>

<tr>
    <td>\S</td>
    <td>This matches any non-whitespace character; this is equivalent to the class
[^ \t\n\r\f\v]</td>
</tr>

<tr>
    <td>\w</td>
    <td>This matches any alphanumeric character; this is equivalent to the class
[a-zA-Z0-9_]</td>
</tr>
    
<tr>
    <td>\W</td>
    <td>This matches any non-alphanumeric character; this is equivalent to the
class [^a-zA-Z0-9_]</td>
</tr>
</tbody>
</table>



<br/><br/>
- Now, we can improve our pattern to find years in a given text a bit:

In [ ]:
txt = """
The first season of Indian Premiere League (IPL) was played in 2008. 
The second season was played in 2009 in South Africa. 
Last season was played in 2018 and won by Chennai Super Kings (CSK).
CSK won the title in 2010 and 2011 as well.
Mumbai Indians (MI) has also won the title 3 times in 2013, 2015 and 2017.
"""
# Retrieve all all odd years
pattern = re.compile(r"\d\d\d[13579]")
matches= pattern.findall(txt)
print(matches)

In [ ]:
highlight(pattern, txt)

- Let us try to find out all special symbols (non-alphanumeric, non-whitespace characters) in our text now.

In [ ]:
txt = """
The first season of Indian Premiere League (IPL) was played in 2008. 
The second season was played in 2009 in South Africa. 
Last season was played in 2018 and won by Chennai Super Kings (CSK).
CSK won the title in 2010 and 2011 as well.
Mumbai Indians (MI) has also won the title 3 times in 2013, 2015 and 2017.
"""
# Find out all special symbols (non-alphanumeric, non-whitespace characters) in text
pattern = re.compile(r"[^\w\s]")
match= pattern.findall(txt)
print(match)

In [ ]:
highlight(pattern, txt)

## Alteration
- Just like character sets are used to match a single character out of several possible characters, **alternation** is used to match a single regular expression out of several possible regular expressions.
- This is accomplished using the pipe symbol `|`.

### Example 1:

In [ ]:
txt = """
the most common conjunctions are and, or and but.
"""
# Find all occurrences of and, or, the in a given text
pattern = re.compile("and|or|the")
match = pattern.findall(txt)
print(match)

In [ ]:
highlight(pattern, txt)

### Quantifiers
- **Quantifiers** are the mechanisms to define how a **character**, **metacharacter**, or **character set** can be **repeated**.
- Here is the list of 4 basic quantifers:

<table style="border: 1px solid black; font-size:15px;">
<thead>
    <th>Symbol</th>
    <th>Name</th>
    <th>Quantification of previous character</th>
</thead>
    
<tbody>
<tr>
    <td>?</td>
    <td>Question Mark</td>
    <td>Optional (0 or 1 repetitions)</td>
</tr>
    
<tr>
    <td>*</td>
    <td>Asterisk</td>
    <td>Zero or more times</td>
</tr>

<tr>
    <td>+</td>
    <td>Plus Sign</td>
    <td>One or more times</td>
</tr>

<tr>
    <td>{n,m}</td>
    <td>Curly Braces</td>
    <td>Between n and m times</td>
</tr>
</tbody>
</table>

### Example 1:

In [ ]:
txt = """
I have 2 dogs. One dog is 1 year old and other one is 2 years old. Both dogs are very cute! 
"""
# Find all the matches for dog and dogs in the given text.
pattern = re.compile("dogs?")
match = pattern.findall(txt)
print(match)

In [ ]:
highlight(pattern, txt)

### Example 2:

In [ ]:
txt = """
file1.txt
file_one.txt
file.txt
fil.txt
file.xml
file-1.txt
"""
# Find all filenames starting with 'file' and ending with '.txt' in the given text.
pattern = re.compile(r"file[\w-]*\.txt")
match= pattern.findall(txt)
print(match)

In [ ]:
highlight(pattern, txt)

### Example 3:

In [ ]:
txt = """
file1.txt
file_one.txt
file09.txt
fil.txt
file23.xml
file.txt
"""
# Find all filenames starting with 'file' followed by 1 or more digits and ending with '.txt' in the given text.
pattern = re.compile(r"file\d+\.txt")
match= pattern.findall(txt)
print(match)

In [ ]:
highlight(pattern, txt)

- We can use the curly brackets syntax here with these modifications:

<table style="border: 1px solid black; font-size:15px;">
<thead>
    <th>Syntax</th>
    <th>Description</th>
</thead>
    
<tbody>
<tr>
    <td>{n}</td>
    <td>The previous character is repeated exactly n times.</td>
</tr>
    
<tr>
    <td>{n,}</td>
    <td>The previous character is repeated at least n times.</td>
</tr>

<tr>
    <td>{,n}</td>
    <td>The previous character is repeated at most n times.</td>
</tr>

<tr>
    <td>{n,m}</td>
    <td>The previous character is repeated between n and m times (both inclusive).</td>
</tr>
</tbody>
</table>

### Example 4:

In [ ]:
txt = """
The first season of Indian Premiere League (IPL) was played in 2008. 
The second season was played in 2009 in South Africa. 
Last season was played in 2018 and won by Chennai Super Kings (CSK).
CSK won the title in 2010 and 2011 as well.
Mumbai Indians (MI) has also won the title 3 times in 2013, 2015 and 2017.
"""
# Find years in the given text
pattern = re.compile(r"2\d{3}")
match= pattern.findall(txt)
print(match)

In [ ]:
highlight(pattern, txt)

### Example 5:

In [ ]:
txt = """
123143
432
5657
4435
54
65111
"""
# Filter out all 4 or more digit numbers.
pattern = re.compile(r"\d{4,}")
match= pattern.findall(txt)
print(match)

In [ ]:
highlight(pattern, txt)

### Example 6:

In [ ]:
txt = """
555-555-5555
555 555 5555
5555555555
"""
# Write a pattern to validate telephone numbers
pattern = re.compile(r"\d{3}[-\s]?\d{3}[-\s]?\d{4}")
match= pattern.findall(txt)
print(match)

In [ ]:
highlight(pattern, txt)

In [ ]:
txt = """
Anita is 22 yrs, 
Priya is 21 yrs, 
Ashish is 21 yrs, 
Prabhat is 19 yrs,
"""
# Create a dictionary of name and age
d= {}
# pattern: ages are in numbers and name starts with uppercase alphabet followed by Lower case.
ages = re.findall(r'\d{1,3}', txt)
name = re.findall(r'[A-Z][a-z]*', txt)
result = dict(zip(name,ages))
print(result)

## Greedy Behaviour
> The greedy behavior of the quantifiers is applied by default in the quantifiers. A greedy quantifier will try to match as much as possible to have the biggest match result possible.


- Let's consider an example.

In [ ]:
txt = """
<html>
<head>
<title>Title</title>
</head>
</html>
"""
pattern = re.compile("<.*>")
match= pattern.findall(txt)
print(match)

In [ ]:
highlight(pattern, txt)

- In above example, one may expect to get 4 matches, i.e. `<html>`, `<head>`, `<title>` and `</title>`. 
    - Instead, we get the longest match, i.e. `<html><head><title>Title</title>`.
    - This particular behaviour (to find longest match) is called **greedy** behaviour.


## Non-Greedy behaviour
> A quantifier marked as reluctant will behave like the exact opposite of the greedy ones. They will try to have the smallest match possible.

- The **non-greedy** (or **reluctant**) behaviour can be requested by adding an extra question mark to the quantifier.
- For example, `??`, `*?` or `+?`

In [ ]:
txt = """
<html>
<head>
<title>Title</title>
</head>
</html>
"""
pattern = re.compile("<.*?>")
match= pattern.findall(txt)
print(match)

In [ ]:
highlight(pattern, txt)

## Boundary Matchers

- Here is a table which shows the list of all boundary matchers available in Python:

<table style="border: 1px solid black; font-size:15px;">
<thead>
    <th>Matcher</th>
    <th>Description</th>
</thead>
    
<tbody>
<tr>
    <td>^</td>
    <td>Matches at the beginning of a line</td>
</tr>
    
<tr>
    <td>$</td>
    <td>Matches at the end of a line</td>
</tr>

<tr>
    <td>\b</td>
    <td>Matches a word boundary</td>
</tr>

<tr>
    <td>\B</td>
    <td>Matches the opposite of \b. Anything that is not a word boundary</td>
</tr>

<tr>
    <td>\A</td>
    <td>Matches the beginning of the input</td>
</tr>

<tr>
    <td>\Z</td>
    <td>Matches the end of the input</td>
</tr>
</tbody>
</table>

- Such identifiers that correspond to a particular position inside of the input are called **Boundary Matchers**.


- **NOTE:** Since `\b` is also an escape sequence for strings in Python, we need to escape it using `\`, i.e. `\\b`,  in order to treat it like a metacharacter for regex matching.

### Example 1:
- Consider a scenario where we want to find all occurances of `and`, `or` and `the` in the given text.

In [ ]:
txt = """
Lorem Ipsum is simply dummy text of the printing and typesetting industry. 
Lorem Ipsum has been the industry's standard dummy text ever since the 1500s, when an unknown printer took a galley of type and scrambled it to make a type specimen book. 
It has survived not only five centuries, but also the leap into electronic typesetting, remaining essentially unchanged. 
It was popularised in the 1960s with the release of Letraset sheets containing Lorem Ipsum passages, 
and more recently with desktop publishing software like Aldus PageMaker including versions of Lorem Ipsum.
"""
pattern = re.compile("and|or|the")
match= pattern.findall(txt)
print(match)

In [ ]:
highlight(pattern, txt)

- There is a slight problem with the above pattern. 
    - `and`, `or`, `the` inside the words are also counted as a match where as we want to find individual strings containing `and`, `or`, `the` only.

### What is the solution??
- Solution is to use this pattern: `\b(and|or|the)\b`
- Where `\b` is a metacharacter that matches at a position that is called a **word boundary**.

In [ ]:
pattern = re.compile("\\b(and|or|the)\\b")
match= pattern.findall(txt)
print(match)

In [ ]:
highlight(pattern, txt)

### Example 2:
- Consider a scenario where we want to find all the lines in the given text which **start** with the pattern `Name:`.

In [ ]:
txt = """
Name:
Age: 0
Roll No.: 15
Grade: S

Name: Ravi
Age: -1
Roll No.: 123 Name: ABC
Grade: K

Name: Ram
Age: N/A
Roll No.: 1
Grade: G
"""

In [ ]:
pattern = re.compile(r"^Name: ?\w*")
match= pattern.findall(txt)
print(match)

> `re.M` (short for `re.MULTILINE`) is a flag which is used to make begin/end `(^, $)` consider each line.

In [ ]:
pattern = re.compile(r"^Name: ?\w*", flags=re.M)
match= pattern.findall(txt)
print(match)

In [ ]:
highlight(pattern, txt)

### Example 3:

In [ ]:
txt = """
Nature's first green is gold,
Her hardest hue to hold.
Her early leaf's a flower;
But only so an hour."""
# Find all the sentences which do not end with a full stop (`.`) in the given text.
pattern = re.compile(r"^.+[^\.]$", flags=re.M)
match= pattern.findall(txt)
print(match)

In [ ]:
highlight(pattern, txt)

### Example 4:

In [ ]:
txt = 'Python is Amazing!'
match = re.findall(r"^\w+",txt)
print(match)

In [ ]:
txt = 'Java is Amazing!'
match = re.findall(r"\w+!$",txt)
print(match)

### Substitution
- Now, we are going to look at a method which will replace all the **leftmost non-overlapping occurrences** of a pattern in a given string and return the new string as result.
- **Syntax:** `sub(repl, string[, count=0])`
- **Syntax:** `re.sub(pattern, repl, string, count=0, flags=0)`
    - `repl` is the replacement string which gets substituted in the place of match
    - `string` is the input text on which substitution takes place.
    - `count` is an optional argument (default is 0) which specifies the max no. of substitutions that can take place.  0 means there is no limit on substitution count.


### Example 1:

In [ ]:
txt = "100 cats, 23 dogs, 3 rabbits"
# Replace all occurances of numbers with a # in the given text.
pattern = re.compile(r"\d+")
newtxt = pattern.sub("#", txt)
print(newtxt)

- **Syntax:** `subn(repl, string[, count=0])`
- **Syntax:** `re.subn(pattern, repl, string, count=0, flags=0)`
    - Returns the substituted string as well as the no. of substitutions.
    - Perform the same operation as sub(), but return a tuple (new_string, number_of_subs_made).

In [ ]:
txt = "100 cats, 23 dogs, 3 rabbits"
# Replace all occurances of numbers with a # in the given text.
pattern = re.compile(r"\d+")
newtxt = pattern.subn("#", txt)
print(newtxt)

### Compilation Flags
- When compiling a pattern string into a pattern object, it's possible to **modify the standard behavior of the patterns** using **Compilation Flags**.
- Multiple compilation flags can be combined using the bitwise OR "|".
- Here is a list of all the complation flags:

<table style="border: 1px solid black; font-size:15px;">
<thead>
    <th>Syntax</th>
    <th>Meaning</th>
</thead>
    
<tbody>
<tr>
    <td>re.IGNORECASE or re.I</td>
    <td>ignore case.</td>
</tr>

<tr>
    <td>re.MULTILINE or re.M</td>
    <td>make begin/end boundary matchers (^, $) consider each line.</td>
</tr>

<tr>
    <td>re.DOTALL or re.S</td>
    <td>make . match newline too.</td>
</tr>

<tr>
    <td>re.UNICODE or re.U</td>
    <td>make {\w, \W, \b, \B} follow Unicode rules.</td>
</tr>

<tr>
    <td>re.LOCALE or re.L</td>
    <td>make {\w, \W, \b, \B} follow locale.</td>
</tr>

<tr>
    <td>re.ASCII or re.A</td>
    <td>make {\w, \W, \b, \B} perform ASCII-only matching.</td>
</tr>

<tr>
    <td>re.VERBOSE or re.X</td>
    <td>allow comment in regex.</td>
</tr>

<tr>
    <td>re.DEBUG</td>
    <td>get information about the compilation pattern.</td>
</tr>
</tbody>
</table>

### 1. re.IGNORECASE or re.I
- This flag makes a regex pattern case-insensitive.


### Example:
- Let's check out an example to find all occurances of `the` and `The` in the given text.

In [ ]:
txt = """
The best thing about regex is that it makes the task of string manipulation so easy.
"""

In [ ]:
pattern = re.compile("the", flags=re.I)
print(pattern)

In [ ]:
highlight(pattern, txt)

### 2. re.MULTILINE or re.M
- This flag is used to make begin/end boundary matchers (`^`, `$`) consider each line of the given text.


- Let's check out an example to find all lines starting with `A`. 

In [ ]:
txt = """
A man was crossing the road.
Suddenly, a car passed before him in a very high speed.
He was terrified
And shocked.
"""

In [ ]:
pattern = re.compile("^A.+", flags=re.M)

In [ ]:
highlight(pattern, txt)

### 3. re.DOTALL or re.S
- The `.` metacharacter matches everything except newline character. If we want to make `.` match newline too, we have to set this flag.


- Let's consider an examle to match all the text after (and including) `car`.

In [ ]:
txt = """
A man was crossing the road.
Suddenly, a car passed before him in a very high speed.
He was terrified and shocked.
"""

In [ ]:
pattern = re.compile("car.+", flags=re.S)

In [ ]:
highlight(pattern, txt)

### 4. re.UNICODE or re.U
- Using this flag, we can make the pattern characters `{\w, \W, \b, \B}` dependent on the Unicode character properties database.

> re.UNICODE is the default flag in Python 3 regex patterns.


- Let's consider an example where we try to work on hindi language.

In [ ]:
txt = "मुझे किताबें पढ़ना बहुत पसंद है।"
pattern = re.compile(r"\w+")
match= pattern.findall(txt)
print(match)

In [ ]:
highlight(pattern, txt)

[Solution](https://stackoverflow.com/questions/12746458/python-unicode-regular-expression-matching-failing-with-some-unicode-characters/12747529#12747529)

In [ ]:
# pip install regex

In [ ]:
# The third-party `regex` module is a drop-in replacement for `re` with fuller
# Unicode support (and atomic groups long before `re` had them).
# It is optional - this cell degrades gracefully if it is not installed.

try:
    import regex

    txt = "मुझे किताबें पढ़ना बहुत पसंद है।"
    pattern = regex.compile(r"\w+")
    print("regex module:", pattern.findall(txt))
    HAS_REGEX = True
except ImportError:
    print("The `regex` module is not installed - run: python -m pip install regex")
    print("Falling back to the standard library:")
    import re

    txt = "मुझे किताबें पढ़ना बहुत पसंद है।"
    pattern = re.compile(r"\w+")
    print("re module   :", pattern.findall(txt))
    HAS_REGEX = False

In [ ]:
# `highlight` expects an `re` pattern, so only call it when we have one
if not HAS_REGEX:
    highlight(pattern, txt)
else:
    print("(highlight() works with re patterns; the regex module has its own API)")

### 5. re.LOCALE or re.L

A **locale** is a set of environment settings defining language, country and character
encoding for an application. This flag makes `\w`, `\W`, `\b` and `\B` depend on the
current locale rather than on Unicode.

> ### 🔴 Version note — `re.LOCALE` is bytes-only in Python 3
>
> ```python
> re.compile(r"\w", re.LOCALE)      # ValueError: cannot use LOCALE flag with a str pattern
> ```
>
> In Python 2, `str` *was* bytes and `re.LOCALE` was how you handled non-ASCII text. In
> Python 3, `str` is Unicode and `re.UNICODE` is already the default — so `re.LOCALE` applies
> **only to `bytes` patterns**, and raises `ValueError` on anything else.
>
> It is also **locale-dependent and therefore not reproducible**: the same code gives
> different results on machines with different `LC_CTYPE` settings.

**You almost certainly want one of these instead:**

| Goal | Use |
|---|---|
| Unicode-aware matching (the default) | Nothing — it is already on |
| ASCII-only matching | **`re.ASCII`** (see below) |
| Byte-level matching with legacy encodings | `re.LOCALE`, on a `bytes` pattern |

The cells below demonstrate the `ValueError`, and then the flag people actually reach for.

In [ ]:
import re

# The full Latin-1 byte range, as a demonstration string
chars = "".join(chr(i) for i in range(256))
print(repr(chars[:80]))
print("...")
print(repr(chars[192:224]))

In [ ]:
import re

# ---- 🔴 re.LOCALE with a str pattern is an error in Python 3 ----
try:
    re.compile(r"\w", re.LOCALE)
except ValueError as exc:
    print("re.compile(r'\\w', re.LOCALE):", exc)

# ---- It is valid only on a BYTES pattern ----
byte_pattern = re.compile(rb"\w", re.LOCALE)
print("\non a bytes pattern:", byte_pattern)
print("matches in b'abc-123':", byte_pattern.findall(b"abc-123"))

print("""
Even here the result depends on the machine's locale settings, which is
why locale-dependent regexes are avoided in portable code.
""")

In [ ]:
import re

# ---- What you usually want instead: re.ASCII ----
text = "".join(chr(i) for i in range(192, 256))     # accented Latin-1 letters

unicode_matches = re.findall(r"\w", text)
ascii_matches = re.findall(r"\w", text, flags=re.ASCII)

print("default (Unicode) matched:", len(unicode_matches), "characters")
print("with re.ASCII     matched:", len(ascii_matches), "characters")
print("\nsample of the Unicode matches:", "".join(unicode_matches[:20]))

highlight(re.compile(r"\w"), "Grüße café 123")

### 6. re.ASCII or re.A

The counterpart to `re.UNICODE`. It makes `\w`, `\W`, `\b`, `\B`, `\d`, `\D`, `\s` and
`\S` match **ASCII characters only**, rather than the full Unicode ranges they use by default.

| Pattern | Default (Unicode) | With `re.ASCII` |
|---|---|---|
| `\w` | Letters and digits in **any** script | `[a-zA-Z0-9_]` |
| `\d` | Any Unicode decimal digit (Arabic, Thai, Devanagari...) | `[0-9]` |
| `\s` | Any Unicode whitespace | `[ \t\n\r\f\v]` |

**Real-world use case:** parsing anything with an ASCII-only specification — identifiers in a
programming language, hex digits, HTTP tokens. Using the Unicode default there would accept
characters the format does not allow.

> Also available as the inline flag `(?a)` at the start of a pattern.

In [ ]:
import re

text = "Grüße 123 café मुझे"

print("pattern r'\\w+' with different flags:")
print("  default (Unicode):", re.findall(r"\w+", text))
print("  re.ASCII         :", re.findall(r"\w+", text, flags=re.ASCII))

print("\npattern r'\\d+':")
print("  default          :", re.findall(r"\d+", "123 ١٢٣ ๑๒๓"))     # Arabic, Thai digits
print("  re.ASCII         :", re.findall(r"\d+", "123 ١٢٣ ๑๒๓", flags=re.ASCII))

# The inline form, scoped to part of a pattern
print("\ninline (?a) flag :", re.findall(r"(?a)\w+", text))

print("""
Use re.ASCII when you deliberately want ASCII-only semantics - parsing
identifiers, hex digits, protocol tokens. Leave it off for human text.
""")

### 7. re.VERBOSE or re.X
- This flag changes the RegEx syntax, to allow you to add annotations in regex. 

- Whitespace within the pattern is ignored, except when in a character class or preceded by an unescaped backslash.

- When a line contains a # neither in a character class or preceded by an unescaped backslash, all characters from the leftmost such # through the end of the line are ignored.

In [ ]:
txt = """
This is a sample text123
"""

In [ ]:
pattern = re.compile(r"\w +")
pattern.findall(txt)

In [ ]:
pattern = re.compile(r"\w +  # find all words", flags=re.X)
pattern.findall(txt)

### 8. re.DEBUG
- This flag when set, gives some information about the compilation pattern.

In [ ]:
pattern = re.compile("\b[a-e7-9]+\b", flags=re.DEBUG)

## Grouping
> Frequently we need to obtain more information than just whether the regex pattern matched or not.

- By placing part of a regular expression inside round brackets or parentheses `(`, `)`, we can **group that part** of the regex pattern together.


### Applications of grouping:

### 1. Apply a quantifier to the entire group:
- For example, `(ab)+` will match one or more repetitions of `ab`.

In [ ]:
txt = "abbbbbabbbb"

In [ ]:
pattern1 = re.compile("ab+")
match = pattern1.findall(txt)
print(match)

In [ ]:
highlight(pattern1, txt)

In [ ]:
pattern2 = re.compile("(ab)+")
match = pattern2.findall(txt)
print(match)

In [ ]:
highlight(pattern2, txt)

### 2. Restrict alternation to part of the RegEx:
### Example 1: 
- `my name is ram|shyam` will match `my name is ram` and `shyam` 
- `my name is (ram|shyam)` will match `my name is ram` and `my name is shyam`.

In [ ]:
txt = """
my name is ram
my name is shyam
"""

In [ ]:
pattern1 = re.compile("my name is ram|shyam")
matches = pattern1.finditer(txt)
for match in matches:
    print(match)

In [ ]:
highlight(pattern1, txt)

In [ ]:
pattern2 = re.compile(r"my name is (ram|shyam)")
matches = pattern2.finditer(txt)
for match in matches:
    print(match)

In [ ]:
highlight(pattern2, txt)

### Example 2:
- Consider one more example now in which we want to search the substrings `What is` and `Who is`.

In [ ]:
txt = """
Who is that guy?
What is the name of that guy?
"""
pattern = re.compile("(What|Who) is")
matches = pattern.finditer(txt)
for match in matches:
    print(match)

In [ ]:
highlight(pattern, txt)

### 3. Group Capturing: Capture the text matched by group:
- Groups indicated with `(`, `)` also capture the **starting** and **ending** index of the text that they match.
- Groups can be retrieved by passing an argument to `group()`, `start()`, `end()`, and `span()` of the `Match` object. 
- Groups are numbered starting with `0`. 
- Group `0` is always present; it captures the whole RegEx pattern, so all `Match` object methods have group `0` as their default argument.

### Example 1:
- Consider an example where we want to parse a date and determine day, month and year.

In [ ]:
txt = "22-04-2020" #Earth day

In [ ]:
pattern = re.compile(r"\d{2}-\d{2}-\d{4}")
match = pattern.findall(txt)
print(match)

In [ ]:
# Lets parse a date and determine day, month and year
pattern = re.compile(r"(\d{2})-(\d{2})-(\d{4})")
match = pattern.match(txt)
print(match)

In [ ]:
# group 0: matches entire regex pattern
print(match.group(0))

In [ ]:
# group 1: match 1st group
print(match.group(1))

In [ ]:
print(match.group(2))

In [ ]:
print(match.group(3))

In [ ]:
print(match.groups()) # Return tuple

In [ ]:
day, month, year = match.groups()

In [ ]:
print(day, month, year)

### Example 2:
- In the given text, find all the patterns with `Name: <some-name>` and extract `<some-name>`. 

In [ ]:
txt = """
Name: Nikhil
Age: 0
Roll No.: 15
Grade: S

Name: Ravi
Age: -1
Roll No.: 123
Grade: K

Name: Ram
Age: N/A
Roll No.: 1
Grade: G
"""

In [ ]:
pattern = re.compile("^Name: (.+)\n", flags=re.M)
match = pattern.findall(txt)
print(match)

In [ ]:
pattern = re.compile("^Name: (.+)\n", flags=re.M)
matches = pattern.finditer(txt)
for match in matches:
    print(match)

> Parentheses cannot be used inside character classes, at least not as metacharacters. When you put a parenthesis in a character class, it is treated as a literal character. So the regex `[(a)b]` matches `a`, `b`, `(`, and `)`.

### Example 3:
- Capture the domain name and unique product id

In [ ]:
txt="""
https://www.amazon.com/dp/B001E4KFG0
https://www.amazon.com/dp/B006K2ZZ7K
https://www.youtube.com/watch?v=xhD4pQlkoJk
https://www.youtube.com/watch?v=Tqsz6fjvhZM
"""

In [ ]:
pattern = re.compile(r'https://www\.([\w-]+)\.com/[\w?]+[/=]([a-zA-Z0-9]+)')
matches = pattern.finditer(txt)
for match in matches:
    print(match.group(1),':',match.group(2))

## Backreferencing
- **Backreferences** in a pattern allow us to specify that the contents of an earlier capturing group must also be found at the current location in the string. 
> For example, `\1` will succeed if the exact contents of group `1` can be found at the current position, and fails otherwise.


### Example 1
- Consider a scenario where we want to find all the duplicated words in the given text.

In [ ]:
txt = """
hello hello
how are you
bye bye
"""
pattern = re.compile(r"(\w+) \1")
match=pattern.findall(txt)
print(match)

> Since Python’s string literals also use a **backslash followed by numbers** to allow including arbitrary characters in a string, backreferences need to be **escaped** so that regex engine gets proper format. We can also use **raw strings** to ignore escaping.


- Here is an example using raw strings.

In [ ]:
pattern = re.compile(r"(\w+) \1")
pattern.findall(txt)

### Example 2
- Consider a scenario where we want to find all dates with the format `dd/mm/yyy` and change them to `yyyy-mm-dd` format. 

In [ ]:
txt = """
today is 23/02/2019.
yesterday was 22/02/2019.
tomorrow is 24/02/2019.
"""

In [ ]:
pattern = re.compile(r"(\d{2})/(\d{2})/(\d{4})")
newtxt = pattern.sub(r"\3-\2-\1", txt)
print(newtxt)

> Backreferences, too, cannot be used inside a character class. The `\1` in a regex like `(a)[\1b]` is either an error or a needlessly escaped literal 1. 

## Named Groups

> Using numbers to refer to groups can be tedious and confusing, and the worst thing is that it doesn't allow you to give meaning or context to the group. That's why we have named groups.

- Instead of referring to groups by numbers, groups can be referenced by a name. Such a group is called a **named group**.
- **Syntax:** `(?P<name>...)`  where `name` is, obviously, the name of the group. 
- Named groups behave exactly like capturing groups, and additionally associate a name with a group.


- Here is a table which shows three different ways to refer to named groups:
    
<table style="border: 1px solid black; font-size:15px;">
<thead>
    <th>Use</th>
    <th>Syntax</th>
</thead>
    
<tbody>
<tr>
    <td>Inside a pattern</td>
    <td>(?P=name)</td>
</tr>
    
<tr>
    <td>In the repl string of the sub operation</td>
    <td>\g&lt;name&gt;</td>
</tr>

<tr>
    <td>In any of the operations of the MatchObject</td>
    <td>match.group('name')</td>
</tr>
</tbody>
</table>


### Example 1
- Consider a scenario where we want to extract the first name and last name of a person.

In [ ]:
txt = "Nikhil Kumar"
pattern = re.compile(r"(?P<first>\w+) (?P<last>\w+)")
match = pattern.match(txt)

In [ ]:
match.group('first')

In [ ]:
match.group('last')

### Example 2
- Now consider the scenario where we want to swap first name and last name in above example.

In [ ]:
txt = "Nikhil Kumar"
pattern = re.compile(r"(?P<first>\w+) (?P<last>\w+)")

In [ ]:
pattern.sub(r"\g<last> \g<first>", txt)

### Example 3
- Consider a scenario where we want to check if a person has same first and last name.

In [ ]:
txt = "Jhonson Jhonson"
pattern = re.compile(r"(?P<first>\w+) (?P=first)")
match= pattern.findall(txt)
print(match)

## Non-Capturing Groups
> There are cases when we want to use groups, but we're not interested in extracting the information, i.e. capturing the matched text inside paranthesis only. An example is **alteration**.


### Example 1:
- Let's consider an example where we want to find the strings `i love cats` or `i love dogs` in the given text.

In [ ]:
txt = """
i hate dragon
i love cats
i love dogs
"""
pattern = re.compile("i love (cats|dogs)")
match= pattern.findall(txt)
print(match)

- As we can see, the group captured part contains only `cats` or `dogs` instead of complete sentences.
- i.e, findall() is returning only the 1st matched group result. But we want complete match.

In [ ]:
for match in pattern.finditer(txt):
    print("Complete regex match (default):", match.group(0))
    print("Match captured by 1st group:", match.group(1))

- Hence, to make a group **non-capturing**, we have to use the syntax `(?:pattern)`.

In [ ]:
pattern = re.compile("i love (?:cats|dogs)")
match = pattern.findall(txt)
print(match)

> After using the new syntax, we have the same functionality as before, but now we're saving resources and the regex is easier to maintain. Note that the group cannot be referenced.

### Example 2:

In [ ]:
txt="""
https://www.facebook.com
http://www.grade-up.edu
http://pm-india.gov.in
"""
# Match all website
pattern = re.compile(r'https?://(?:www\.)?[\w-]+\.(?:com|edu|in|gov)(?:\.\w+)?') 
match = pattern.findall(txt)
print(match)

### Example 3:

In [ ]:
txt="""
mrhappy24@gmail.com
mr_happy24@yahoo.in
mr-happy24@adobe-india.co
"""
# Match all mailid
pattern = re.compile(r'[\w-]+@[\w-]+\.+(?:com|edu|in|co)')
match = pattern.findall(txt)
print(match)

### Example 4:

In [ ]:
txt="""
Mr. Nikhil
Mr Nikhil
Ms Deepa
Mrs. Deepa
Mr. D
"""
# Match all names
pattern = re.compile(r'M(?:r|s|rs)\.?\s[A-Z]\w*')
match = pattern.findall(txt)
print(match)

## Zero-Width Assertions
- Characters which indicate positions rather than actual content are called **zero-width assertions**.
- For instance, the caret symbol (`^`) is a representation of the beginning of a line or the dollar sign (`$`) for the end of a line. 
- They effectively do assertion without consuming characters; they just return a positive or negative result of the match.
- A more powerful kind of **zero-width assertion** is **look around**, a mechanism with which it is possible to match a certain previous (**look behind**) or ulterior (**look ahead**) value to the current position.


## Look around
- **Look around** is a simple mechanism which during the matching process, at the current position, looks forward (or behind, depends on type of lookaround used) to see if **some** pattern matches before continuing with the actual match.

- The most important thing to understand here is that **look around** mechanism consists of 2 parts:
    - **actual expression**: an expression whose match constitutes the final **result**.
    - **non-consuming expression**: an expression whose match is evaluated before the actual expression, just to see if it can succeed. It is **not actually consumed** by the regex engine.
        - If the non-consuming match **succeeds**, the regex engine forgets about this non-consuming expression and starts evaluating the next character from the current position of the actual expression. 
        - If the non-consuming match **does not succeed**, we simply move to next character of the given text and repeat the whole match process again.

- There are 2 main categories of **look around**  which, in turn, have 2 sub-categories each.
<img src='./Image/9.1 Image g.png'>


- Let's explore each one of them one by one.

## Look ahead
- **Look ahead** mechanism checks the match for a non-consuming expression **ahead** of a given pattern.


### Positive look ahead
- **Positive look ahead** will succeed if the passed non-consuming expression **does match** against the forthcoming input.
- The syntax is `A(?=B)` where `A` is the **actual expression** and `B` is the **non-consuming expression**. 


- Let's assume that we want to find a match for `love` in the given text only if it is followed by `regex`.

In [ ]:
txt = "i love python, i love regex"

In [ ]:
pattern = re.compile('love regex')
match = pattern.search(txt)
print(match.span())

In [ ]:
pattern.findall(txt)

In [ ]:
highlight(pattern, txt)

- As we can see, a total of 10 (index 17 to 27) characters, i.e. `love regex` are consumed to search for the given pattern in the text.


- Now consider the regex pattern `love(?=\sregex)`.

In [ ]:
pattern = re.compile(r"love(?=\sregex)")
match = pattern.search(txt)
print(match.span())

In [ ]:
highlight(pattern, txt)

- Now, using **positive look ahead** mechanism, we consumed only 4 (index 17 to 21) characters are consumed for the match.


- Let us check out another example to find all words in given text which are followed by `.` or `,`.

In [ ]:
txt = "My favorite colors are red, green, and blue."
pattern = re.compile(r"\w+(?=,|\.)")
match = pattern.findall(txt)
print(match)

In [ ]:
highlight(pattern, txt)

### Negative look ahead
- **Negative look ahead** will succeed if the passed non-consuming expression **does not match** against the forthcoming input.
- The syntax is `A(?!B)` where `A` is the **actual expression** and `B` is the **non-consuming expression**. 


- Let's assume that we want to find a match for `love` in the given text only if it is NOT followed by `regex`.

In [ ]:
txt = "i love python, i love regex"
pattern = re.compile(r"love(?!\sregex)")
match = pattern.findall(txt)
print(match)

In [ ]:
highlight(pattern, txt)

## Look behind
- **Look behind** mechanism checks the match for a non-consuming expression **behind** a given pattern.


### Positive look behind
- **Positive look behind** will succeed if the passed non-consuming expression **does match** against the forthcoming input.
- The syntax is `(?<=B)A` where `A` is the **actual expression** and `B` is the **non-consuming expression**. 


- Let's assume that we want to find a match for `regex` in the given text only if it is succeeded by `love` or `hate`.

In [ ]:
txt = "love regex or hate regex, can't ignore regex"
pattern = re.compile(r"(?<=(?:love|hate)\s)regex")
match = pattern.findall(txt)
print(match)

In [ ]:
highlight(pattern, txt)

### Negative look behind
- **Negative look behind** will succeed if the passed non-consuming expression **does not match** against the forthcoming input.
- The syntax is `(?<!B)A` where `A` is the **actual expression** and `B` is the **non-consuming expression**. 


- Let's assume that we want to find a match for `regex` in the given text if it is not followed by `love` or `hate`.

In [ ]:
txt = "love regex or hate regex, can't ignore regex"
pattern = re.compile(r"(?<!(?:love|hate)\s)regex")
match = pattern.findall(txt)
print(match)

In [ ]:
highlight(pattern, txt)

In [ ]:
youtube_titles_views = [("How to Tell if We're Beating COVID-19", 2200000), 
                        ("Extreme Closet Clean Out", 326000), 
                        ("This is $1,000,000 in Food", 8800000),
                        ("How To Tell If Someone Truly Loves You ", 2800000), 
                        ("How to Tell Real Gold from Fake", 2300000), 
                        ("Extreme living room transformation ", 25000)]

In [ ]:
first_words = []
views = []
for title in youtube_titles_views:
    first_words.append(re.findall(r"^\w+",title[0])[0])
    views.append(title[1])
    
print(first_words)
print(views)

In [ ]:
import pandas as pd
df = pd.DataFrame({'first_words': first_words, 'views':views})
df = df.groupby('first_words')['views'].mean().sort_values(ascending = False)
print(df)

---

## Catastrophic backtracking (ReDoS)

Everything so far has been about *what* a pattern matches. This section is about how long it
takes — because a regex that looks harmless can take **longer than the age of the universe**
on a 30-character string.

### The mechanism

Python's `re` is a **backtracking** engine. When a match fails, it backs up and tries a
different way of splitting the input between the quantifiers. Usually that is a handful of
attempts. But when one quantifier is **nested inside another** — `(a+)+`, `(a|a)*`,
`(\d+)*` — the number of ways to split the string grows **exponentially**.

```
(a+)+$   against  "aaaaaaaaaaaaaaaaaaaaaaaaaaX"
```

There is no match (the `X` guarantees failure), but before giving up the engine tries every
possible way of partitioning those `a`s among the two quantifiers — 2ⁿ of them.

### The warning signs

| Shape | Example | Why |
|---|---|---|
| Nested quantifiers | `(a+)+`, `(a*)*` | Exponential partitions |
| Alternation with overlap, quantified | `(a\|a)*`, `(\w\|\d)+` | Both branches match the same text |
| Adjacent quantifiers on the same class | `\d+\d+` | Ambiguous split point |
| Quantified group ending in an optional | `(\s*\w*)+` | Empty matches multiply |

> **Why this is a security issue.** If your server applies a vulnerable regex to user input —
> validating an email, parsing a User-Agent header — an attacker sends one crafted string and
> pins a CPU core. That is a **ReDoS** (Regular expression Denial of Service) attack, and it
> has taken down real production systems.

In [ ]:
import re
import time

# ---- A vulnerable pattern: a quantifier inside a quantifier ----
vulnerable = re.compile(r"(a+)+$")

print("length | time to FAIL")
print("-------+--------------")
for n in range(16, 25):
    text = "a" * n + "X"          # the X guarantees no match
    start = time.perf_counter()
    vulnerable.search(text)
    elapsed = time.perf_counter() - start
    print(f"  {n:>4} | {elapsed * 1000:9.2f} ms")
    if elapsed > 2:
        print("       | ...stopping - each extra character DOUBLES the time")
        break

print("""
Every additional 'a' doubles the work. At n=30 this is minutes; at n=40,
days. The string is 40 characters long.
""")

# ---- The same input against a SAFE pattern ----
safe = re.compile(r"a+$")
start = time.perf_counter()
safe.search("a" * 5000 + "X")
print(f"safe pattern, 5000 chars: {(time.perf_counter() - start) * 1000:.3f} ms")

### The three fixes

> **Version note:** **atomic groups `(?>...)` and possessive quantifiers `*+`, `++`, `?+`
> were added to `re` in Python 3.11**. Before that you needed the third-party `regex` module.

| Fix | How | When |
|---|---|---|
| **1. Restructure the pattern** | Remove the ambiguity: `(a+)+` → `a+` | Always try this first |
| **2. Atomic group** | `(?>a+)` — once matched, never backtracked into | The general tool |
| **3. Possessive quantifier** | `a++` — same idea, shorter | Simple cases |

An **atomic group** tells the engine: *"match this as far as you can, and never give any of
it back."* That single rule collapses the exponential search space to a linear one.

Possessive quantifiers are shorthand: `a++` is exactly `(?>a+)`.

In [ ]:
import re
import time

TEXT = "a" * 30 + "X"


def timed(pattern: str, text: str, label: str, limit: float = 3.0) -> None:
    compiled = re.compile(pattern)
    start = time.perf_counter()
    try:
        result = compiled.search(text)
    except Exception as exc:                 # noqa: BLE001 - demo only
        print(f"  {label:<32} {type(exc).__name__}")
        return
    elapsed = time.perf_counter() - start
    verdict = "match" if result else "no match"
    print(f"  {label:<32} {elapsed * 1000:9.3f} ms  ({verdict})")


print(f"input: 30 x 'a' followed by 'X'  (len {len(TEXT)})\n")

# 1. Restructured - the nesting was never needed
timed(r"a+$", TEXT, "a+$  (restructured)")

# 2. Atomic group (3.11+)
timed(r"(?>a+)+$", TEXT, "(?>a+)+$  (atomic group)")

# 3. Possessive quantifier (3.11+)
timed(r"(a+)++$", TEXT, "(a+)++$  (possessive)")

print("""
  (?>a+)+$ and (a+)++$ still express the nested shape, but the engine is
  forbidden from backtracking into the group - so it fails immediately
  instead of exploring 2^30 partitions.
""")

# ---- What atomic actually changes ----
print("atomic vs greedy on the SAME input:")
print("  r'a*a'  on 'aaa' ->", re.search(r"a*a", "aaa"), " <- greedy gives back one 'a'")
print("  r'(?>a*)a' on 'aaa' ->", re.search(r"(?>a*)a", "aaa"), " <- atomic keeps them all, so it fails")

print("""
That second result is the trade-off: atomic groups are FASTER but can change
what matches. Restructuring the pattern is always the safest fix.
""")

---

## Performance, and when not to use a regex

### Compile once, use many times

`re.compile()` parses the pattern and builds a state machine. Calling `re.search(pattern, s)`
does that too — but `re` keeps an internal cache of the last 512 patterns, so the cost is
usually hidden.

Compile explicitly when:
- The pattern is used in a **loop**
- The pattern is built at run time (so the cache may thrash)
- You want the pattern **named** and defined near the top of the module

### ⚠️ The bigger win: don't use a regex at all

A regex is the right tool for **structured pattern matching**. For simple string operations
the built-in `str` methods are shorter, clearer and several times faster.

| You want | Don't | Do |
|---|---|---|
| Does it start with X? | `re.match(r"^X", s)` | `s.startswith("X")` |
| Does it end with X? | `re.search(r"X$", s)` | `s.endswith("X")` |
| Does it contain X? | `re.search(r"X", s)` | `"X" in s` |
| Split on a fixed string | `re.split(r",", s)` | `s.split(",")` |
| Replace a fixed string | `re.sub(r"a", "b", s)` | `s.replace("a", "b")` |
| Strip characters | `re.sub(r"^\s+\|\s+$", "", s)` | `s.strip()` |

Reach for `re` when you need **alternation, quantifiers, character classes, groups or
anchors** — that is, when the thing you are matching genuinely has structure.

In [ ]:
import re
import timeit

text = "The quick brown fox jumps over the lazy dog" * 20

# ---- Compiled vs module-level ----
compiled = re.compile(r"\bqu\w+")

t_module = timeit.timeit(lambda: re.findall(r"\bqu\w+", text), number=20_000)
t_compiled = timeit.timeit(lambda: compiled.findall(text), number=20_000)

print(f"re.findall(pattern, s) : {t_module * 1000:7.1f} ms")
print(f"compiled.findall(s)    : {t_compiled * 1000:7.1f} ms")
print(f"compiled is {t_module / t_compiled:.2f}x faster")
print("  (the gap is small because re caches compiled patterns internally)")


# ---- Regex vs str methods for simple jobs ----
print("\nsimple operations - regex vs str:")

cases = [
    ("startswith", lambda: text.startswith("The"),        lambda: re.match(r"The", text)),
    ("contains",   lambda: "fox" in text,                  lambda: re.search(r"fox", text)),
    ("split",      lambda: text.split(" "),                lambda: re.split(r" ", text)),
    ("replace",    lambda: text.replace("fox", "cat"),     lambda: re.sub(r"fox", "cat", text)),
]

print(f"  {'operation':<12} {'str':>10} {'regex':>10}   ratio")
for label, str_fn, re_fn in cases:
    t_str = timeit.timeit(str_fn, number=20_000)
    t_re = timeit.timeit(re_fn, number=20_000)
    print(f"  {label:<12} {t_str * 1000:9.1f}ms {t_re * 1000:9.1f}ms   {t_re / t_str:5.1f}x")

print("""
str methods win on both speed and readability for fixed strings.
Use re when the thing you are matching has STRUCTURE.
""")

---

## Common Mistakes & Pitfalls

1. 🔴 **Not using raw strings.** `re.compile("\\w+")` in a normal string is a `SyntaxWarning` on 3.12+ and is documented to become an error. Always `r"..."`.
2. **Confusing `match`, `search` and `fullmatch`.** `match` anchors at the start only; `search` scans anywhere; `fullmatch` requires the whole string.
3. **Assuming `match` implies `^...$`.** It anchors the start, not the end — `re.match(r"\\d+", "123abc")` succeeds.
4. 🔴 **Nested quantifiers** like `(a+)+`. Exponential backtracking; a ReDoS waiting to happen.
5. **Using `.` expecting it to match a newline.** It does not, unless you pass `re.DOTALL`.
6. **Greedy `.*` swallowing too much.** Use `.*?` when you want the shortest match.
7. **`findall` with groups returning tuples, not whole matches.** With one group it returns that group; with several, tuples. Use `finditer` when you want `Match` objects.
8. **Passing `maxsplit`/`count` positionally** to `re.split`/`re.sub` — deprecated in 3.13.
9. **`re.LOCALE` with a `str` pattern** — `ValueError`. It is bytes-only in Python 3.
10. **Not escaping user input** interpolated into a pattern. Use `re.escape()`.
11. **Using a regex to parse HTML, JSON or CSV.** Use a real parser (**8.2**, **8.3**).

## Best Practices

- **Always** write patterns as raw strings: `r"\\d+"`.
- Compile patterns used in loops, and name them as module-level constants.
- Use `re.VERBOSE` with comments for any pattern longer than about 30 characters.
- Use **named groups** `(?P<name>...)` and `match.groupdict()` instead of numbered groups.
- Use `re.fullmatch()` for validation — it is what you almost always mean.
- Use the walrus operator: `if m := pattern.search(s):`.
- Use `re.escape()` on any literal text you interpolate into a pattern.
- Prefer `str` methods for fixed-string work — they are faster and clearer.
- Test patterns against inputs that should **fail**, not just ones that should match.
- Watch for nested quantifiers; restructure, or use an atomic group (3.11+).

## Practice Exercises

Try these before moving on.

1. Validate an email with `fullmatch`, then explain why a 'complete' email regex is a bad idea.
2. Write a pattern with named groups that parses `2024-03-15 14:30:00` and returns a `groupdict()`.
3. Rewrite a long pattern using `re.VERBOSE` with comments on each part.
4. Time `(a+)+$` against `a+$` for inputs of length 20, 22 and 24. Plot the shape.
5. Fix a catastrophic pattern three ways: restructured, atomic group, possessive quantifier.
6. Use `re.sub` with a **function** replacement to convert `snake_case` to `camelCase`.
7. Use `re.escape()` to build a pattern from user input safely, and show what breaks without it.
8. Take five regexes from this notebook and rewrite each with `str` methods where possible.
9. Extract all URLs from a block of text, then explain three cases your pattern gets wrong.